In [ ]:
from google.colab import files
import pandas as pd

print("Apni bank.csv file upload karo:")
uploaded = files.upload()
DATA_PATH = list(uploaded.keys())[0]

with open(DATA_PATH, newline='', encoding='utf-8', errors='replace') as f:
    first = f.readline()
    sep = ';' if first.count(';') > first.count(',') else ','
    df_preview = pd.read_csv(DATA_PATH, sep=sep)

    print(f"Loaded: {DATA_PATH} | separator='{sep}' | rows={len(df_preview)}, cols={df_preview.shape[1]}")
    df_preview.head()


Apni bank.csv file upload karo:


Saving bank.csv to bank.csv
Loaded: bank.csv | separator=',' | rows=11162, cols=17


In [ ]:
import json
import os
import re
from datetime import datetime

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

EMAIL_RE = re.compile(r"^[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}$")
PHONE_RE = re.compile(r"^\+?\d[\d\s().-]{6,18}\d$")
UK_POSTCODE_RE = re.compile(r"^[A-Za-z]{1,2}\d[A-Za-z\d]?\s*\d[A-Za-z]{2}$")
URL_RE = re.compile(r"^(https?://|www\.)[\w.-]+(\.[a-zA-Z]{2,})")

PII_KEYWORDS = {
    "email": ["email", "e-mail", "mail"],
    "phone": ["phone", "mobile", "contact", "tel"],
    "name": ["name", "first_name", "last_name", "surname", "fullname"],
    "address": ["address", "addr", "street", "city", "postcode", "zip"],
    "dob": ["dob", "birth", "age"],
    "id": ["ssn", "passport", "national_id", "nic"],
}

SEMANTIC_PATTERNS = {
    "email": lambda s: bool(EMAIL_RE.match(str(s))),
    "phone": lambda s: bool(PHONE_RE.match(str(s))),
    "postcode": lambda s: bool(UK_POSTCODE_RE.match(str(s))),
    "url": lambda s: bool(URL_RE.match(str(s))),
}

def _sample(series, n=500):
    vals = series.dropna()
    return vals.sample(n=min(n, len(vals)), random_state=42) if len(vals) > n else vals

def detect_semantic_meaning(series):
    vals = _sample(series)
    if len(vals) == 0:
        return None
    if pd.api.types.is_string_dtype(series.dtype):
        try:
            parsed = pd.to_datetime(vals, errors="raise", format="mixed")
            if parsed.notna().mean() >= 0.8:
                return "date"
        except (ValueError, TypeError):
            pass
    for meaning, test in SEMANTIC_PATTERNS.items():
        hits = sum(test(v) for v in vals)
        if hits / len(vals) >= 0.8:
            return meaning
    return None

def infer_column_type(series):
    vals = _sample(series)
    non_null = series.dropna()
    result = {"pandas_dtype": str(series.dtype), "logical_type": None, "is_mixed_type": False}
    if len(non_null) == 0:
        result["logical_type"] = "empty"
        return result
    if pd.api.types.is_numeric_dtype(series):
        result["logical_type"] = "integer" if pd.api.types.is_integer_dtype(series) else "float"
        return result
    if pd.api.types.is_datetime64_any_dtype(series):
        result["logical_type"] = "date"
        return result
    if pd.api.types.is_bool_dtype(series):
        result["logical_type"] = "boolean"
        return result
    type_hits = {"int": 0, "float": 0, "date": 0, "bool": 0, "str": 0}
    for v in vals:
        s = str(v).strip()
        try:
            int(s); type_hits["int"] += 1; continue
        except ValueError:
            pass
        try:
            float(s); type_hits["float"] += 1; continue
        except ValueError:
            pass
        if s.lower() in ("true", "false"):
            type_hits["bool"] += 1; continue
        try:
            pd.to_datetime(s, format="mixed"); type_hits["date"] += 1; continue
        except (ValueError, TypeError):
            pass
        type_hits["str"] += 1
    total = sum(type_hits.values())
    result["logical_type"] = max(type_hits, key=type_hits.get)
    mixed = [t for t, c in type_hits.items() if c / total >= 0.05]
    result["is_mixed_type"] = len(mixed) > 1
    result["type_composition"] = {k: round(c / total, 3) for k, c in type_hits.items() if c}
    return result

def _is_id_like(name, nunique, nrows):
    lname = name.lower()
    name_hint = any(k in lname for k in ["id", "uuid", "code", "number"])
    return name_hint and nunique / nrows >= 0.9

def profile_column(series, col_name):
    n = len(series)
    non_null = series.dropna()
    nunique = int(non_null.nunique())
    profile = {
        "dtype": str(series.dtype),
        "missing_count": int(series.isna().sum()),
        "missing_pct": round(float(series.isna().mean() * 100), 2),
        "unique_count": nunique,
        "cardinality_ratio": round(nunique / n, 4) if n else 0,
        "is_id_like": _is_id_like(col_name, nunique, n),
    }
    profile.update(infer_column_type(series))
    profile["semantic_meaning"] = detect_semantic_meaning(series)
    if pd.api.types.is_numeric_dtype(series):
        desc = non_null.describe()
        q1, q3 = desc["25%"], desc["75%"]
        iqr = q3 - q1
        outliers = non_null[(non_null < q1 - 1.5 * iqr) | (non_null > q3 + 1.5 * iqr)]
        profile["statistics"] = {
            "mean": round(float(desc["mean"]), 4),
            "std": round(float(desc["std"]), 4) if not pd.isna(desc["std"]) else 0.0,
            "min": round(float(desc["min"]), 4),
            "max": round(float(desc["max"]), 4),
            "quartiles": {k: round(float(desc[k]), 4) for k in ["25%", "50%", "75%"]},
            "skew": round(float(non_null.skew()), 4) if nunique > 2 else 0.0,
            "iqr_outlier_count": int(len(outliers)),
        }
    elif profile["logical_type"] in (None, "str") and nunique <= 50:
        profile["top_values"] = non_null.astype(str).value_counts().head(10).to_dict()
    return profile

def detect_pii(columns):
    pii = []
    for name, prof in columns.items():
        lname = name.lower()
        hit = None
        for kind, keywords in PII_KEYWORDS.items():
            if any(k in lname for k in keywords):
                hit = kind
                break
        if prof.get("semantic_meaning") in ("email", "phone", "postcode"):
            hit = prof["semantic_meaning"]
        if hit:
            pii.append({"column": name, "pii_type": hit, "reason": "name_keyword" if hit in PII_KEYWORDS else "value_pattern"})
    return pii

def run_rule_engine(df, columns):
    flags = {"suspicious_columns": [], "pii_columns": detect_pii(columns), "warnings": []}
    if df.columns.duplicated().any():
        flags["warnings"].append("Duplicate column names detected")
    for name, prof in columns.items():
        issues = []
        if prof["missing_pct"] > 50:
            issues.append(f"very high missingness ({prof['missing_pct']}%)")
        if prof["unique_count"] <= 1 and len(df) > 1:
            issues.append("constant column (single value)")
        if prof.get("is_mixed_type"):
            issues.append("mixed data types in one column")
        if prof["is_id_like"]:
            issues.append("ID-like column (high cardinality)")
        if prof.get("iqr_outlier_count", 0) > len(df) * 0.05:
            issues.append(f"heavy outliers ({prof['iqr_outlier_count']} rows via IQR)")
        if issues:
            flags["suspicious_columns"].append({"column": name, "issues": issues})
    return flags

def make_visuals(df, out_dir):
    os.makedirs(out_dir, exist_ok=True)
    paths = {}
    if df.isna().any().any():
        fig, ax = plt.subplots(figsize=(max(6, len(df.columns) * 0.6), 4))
        sample = df.head(1000)
        ax.imshow(sample.isna().T.values, aspect="auto", cmap="Reds", interpolation="nearest")
        ax.set_yticks(range(len(sample.columns)))
        ax.set_yticklabels(sample.columns, fontsize=7)
        ax.set_title("Missing-Value Matrix (rows sampled, red = missing)")
        ax.set_xlabel("row index")
        fig.tight_layout()
        p = os.path.join(out_dir, "missing_heatmap.png")
        fig.savefig(p, dpi=120)
        plt.close(fig)
        paths["missing_heatmap"] = p
    num = df.select_dtypes(include=np.number)
    if num.shape[1] > 0:
        fig, ax = plt.subplots(figsize=(max(6, num.shape[1] * 0.8), 4))
        ax.boxplot([num[c].dropna() for c in num.columns], tick_labels=num.columns)
        ax.set_title("Outlier Distribution (numeric columns)")
        ax.tick_params(axis="x", rotation=45, labelsize=7)
        fig.tight_layout()
        p = os.path.join(out_dir, "outlier_distribution.png")
        fig.savefig(p, dpi=120)
        plt.close(fig)
        paths["outlier_distribution"] = p
    if num.shape[1] > 1:
        corr = num.corr()
        fig, ax = plt.subplots(figsize=(max(5, num.shape[1] * 0.6), max(4, num.shape[1] * 0.5)))
        im = ax.imshow(corr.values, cmap="coolwarm", vmin=-1, vmax=1)
        ax.set_xticks(range(len(corr.columns)))
        ax.set_yticks(range(len(corr.columns)))
        ax.set_xticklabels(corr.columns, rotation=45, ha="right", fontsize=7)
        ax.set_yticklabels(corr.columns, fontsize=7)
        for i in range(len(corr.columns)):
            for j in range(len(corr.columns)):
                ax.text(j, i, f"{corr.values[i, j]:.2f}", ha="center", va="center", fontsize=6)
        fig.colorbar(im)
        ax.set_title("Correlation Heatmap")
        fig.tight_layout()
        p = os.path.join(out_dir, "correlation_heatmap.png")
        fig.savefig(p, dpi=120)
        plt.close(fig)
        paths["correlation_heatmap"] = p
    return paths

In [ ]:
def profile_dataset(input_path, output_dir):
    df = pd.read_csv(input_path, sep=None, engine="python")
    os.makedirs(output_dir, exist_ok=True)
    columns = {col: profile_column(df[col], col) for col in df.columns}
    num = df.select_dtypes(include=np.number)
    corr = num.corr().round(4).to_dict() if num.shape[1] > 1 else {}
    report = {
        "dataset": {
            "file": os.path.basename(input_path),
            "rows": int(len(df)),
            "columns": int(df.shape[1]),
            "memory_mb": round(df.memory_usage(deep=True).sum() / 1e6, 2),
        },
        "generated_at": datetime.now().isoformat(timespec="seconds"),
        "columns": columns,
        "correlation_matrix": corr,
        "rules": run_rule_engine(df, columns),
        "visuals": {},
    }
    report["visuals"] = {k: os.path.relpath(v, output_dir)
                         for k, v in make_visuals(df, os.path.join(output_dir, "visuals")).items()}
    out_file = os.path.join(output_dir, "profiling_report.json")
    with open(out_file, "w") as f:
        json.dump(report, f, indent=2)
    print(f"Report written to {out_file}")
    return report

report = profile_dataset(DATA_PATH, 'results')
print()
print('--- Suspicious columns ---')
for s in report['rules']['suspicious_columns']:
    print(' ', s['column'], '->', '; '.join(s['issues']))
print()
print('PII flagged:', [(p['column'], p['pii_type']) for p in report['rules']['pii_columns']])

Report written to results/profiling_report.json

--- Suspicious columns ---

PII flagged: [('age', 'dob'), ('contact', 'phone')]


In [ ]:
suspicious = []
total_rows = report['dataset']['rows']

for col, prof in report['columns'].items():
    stats = prof.get('statistics', {})
    issues = []
    if stats.get('iqr_outlier_count', 0) > total_rows * 0.05:
        issues.append(f"heavy outliers ({stats['iqr_outlier_count']} rows via IQR)")
    if prof['missing_pct'] > 50:
        issues.append(f"very high missingness ({prof['missing_pct']}%)")
    if prof.get('is_mixed_type'):
        issues.append("mixed data types in one column")
    if prof['is_id_like']:
        issues.append("ID-like column (high cardinality)")
    if issues:
        suspicious.append({'column': col, 'issues': issues})

report['rules']['suspicious_columns'] = suspicious

print('--- Suspicious columns ---')
for s in suspicious:
    print(' ', s['column'], '->', '; '.join(s['issues']))

json.dump(report, open('results/profiling_report.json', 'w'), indent=2)
print()
print('Report update ho gayi!')

--- Suspicious columns ---
  balance -> heavy outliers (1055 rows via IQR)
  duration -> heavy outliers (636 rows via IQR)
  campaign -> heavy outliers (601 rows via IQR)
  pdays -> heavy outliers (2750 rows via IQR)
  previous -> heavy outliers (1258 rows via IQR)

Report update ho gayi!
